# w7v2_02C — run DAF on the shared page sets

Runs **DAF** (Faster R-CNN version, 3 weight files: `srnet`, `anytext`, `diffste`) on every unique image of
`PAGES_ZIP_NAME.zip` at 3 sizes (`shrink512`, `default`, `full`) and saves the team's STANDARD result maps.
The only thing to send Abby at the end is the Drive link to `daf_results.zip`.

**How to run**
1. Runtime → Change runtime type → **GPU**.
2. Put `PAGES_ZIP_NAME.zip`, `srnet.pth`, `anytext.pth`, `diffste.pth` in `PATH_TO_DRIVE_FOLDER/`.
3. Run all with `SMOKE = True` first (3 pages per set). Check sections 4, 7 and 8.
4. Set `SMOKE = False`, Run all again. If Colab disconnects: reconnect and Run all — finished files are skipped.

DAF itself runs in a **separate Python 3.9 environment** (section 3) as a script; this notebook only prepares
inputs, starts the script, and turns its boxes into maps.

## 1 · CONFIG
**Why:** every setting lives in one place, so a re-run uses exactly the same choices, and `config.json` in
`results/` records them for Abby. Results go straight to Google Drive so nothing is lost if Colab disconnects.
Keep `SMOKE = True` for the first run: it processes only 3 pages per set so mistakes show up in minutes.

In [ ]:
# Standard Python tools used across the notebook
import os, sys, json, time, glob, math, shutil, hashlib, zipfile, random, subprocess
# Number and table tools
import numpy as np
import pandas as pd

# True when this notebook runs inside Google Colab
IN_COLAB = "google.colab" in sys.modules
# Mount Google Drive (approve the pop-up) so inputs and results live on Drive
if IN_COLAB:
    # The Colab helper that connects Drive
    from google.colab import drive
    # Drive appears under /content/drive
    drive.mount("/content/drive")

# SMOKE=True: quick test on a few pages. Set to False for the real run.
SMOKE = False
# Pages per set used in SMOKE mode
SMOKE_N = 3
# Fixed seed (nothing here is random, but we record it anyway)
SEED = 42
# The three DAF weight files, by name
WEIGHTS = ["srnet", "anytext", "diffste"]
# The three input sizes
SIZES = ["shrink512", "default", "full"]
# Piece size for "full": length along the page's long side
PIECE_LONG = 1536
# Piece size for "full": length along the page's short side
PIECE_SHORT = 1024
# Long side used by "shrink512"
SHRINK_LONG = 512
# Score threshold used only for printing
THRESH = 0.5
# Where the weights were downloaded from (pths/dfpn/ = Faster R-CNN, the only kind these configs load)
WEIGHT_SOURCE = "ModelScope DanceText/DAF, folder pths/dfpn/ (Faster R-CNN DAF)"

# The Drive folder with the inputs; results/ is written inside it
DRIVE_DIR = os.environ.get("W7_DAF_DRIVE_DIR", "PATH_TO_DRIVE_FOLDER")
# Fast local disk of the Colab machine
LOCAL_DIR = os.environ.get("W7_DAF_LOCAL_DIR", "/content")
# Where the page sets are unzipped (local disk is much faster than Drive)
SETS_DIR = os.path.join(LOCAL_DIR, "w7_sets")
# Where the Python 3.9 environment for DAF is built
ENV_DIR = os.path.join(LOCAL_DIR, "daf_env")
# The Python 3.9 interpreter inside that environment
ENV_PY = os.path.join(ENV_DIR, "bin", "python")
# Where the OSTF (DAF) code is cloned
REPO_DIR = os.path.join(LOCAL_DIR, "OSTF")
# Where resized pages and pieces are saved (local disk)
INPUT_DIR = os.path.join(LOCAL_DIR, "daf_inputs")
# The DAF inference script written in section 4
SCRIPT = os.path.join(LOCAL_DIR, "daf_infer.py")
# Final results for Abby (on Drive)
RESULTS_DIR = os.path.join(DRIVE_DIR, "results")
# Raw DAF boxes, one JSON per page (on Drive, so a disconnect loses nothing; not zipped)
RAW_DIR = os.path.join(DRIVE_DIR, "raw_daf")

# Set the seeds
random.seed(SEED)
# Same for numpy
np.random.seed(SEED)
# Stop early if the Drive folder is missing
if not os.path.isdir(DRIVE_DIR):
    # Clear message instead of a confusing error later
    raise RuntimeError(f"STOP: Drive folder not found: {DRIVE_DIR}. Create it and put the zip + 3 .pth files in it.")
# Make the output folders
for d in [RESULTS_DIR, RAW_DIR, INPUT_DIR]:
    # exist_ok: re-runs are fine
    os.makedirs(d, exist_ok=True)

# All settings in one dictionary
CONFIG = {"SMOKE": SMOKE, "SMOKE_N": SMOKE_N, "SEED": SEED, "WEIGHTS": WEIGHTS, "SIZES": SIZES,
          "PIECE_LONG": PIECE_LONG, "PIECE_SHORT": PIECE_SHORT, "SHRINK_LONG": SHRINK_LONG,
          "THRESH": THRESH, "WEIGHT_SOURCE": WEIGHT_SOURCE, "DRIVE_DIR": DRIVE_DIR}
# Save them into results/
with open(os.path.join(RESULTS_DIR, "config.json"), "w") as f:
    # Pretty JSON
    json.dump(CONFIG, f, indent=2)

# Show the settings
print(json.dumps(CONFIG, indent=2))
# Show where results go
print("results ->", RESULTS_DIR)
# Big warning while in SMOKE mode
if SMOKE:
    # Hard to miss
    print("\n" + "!" * 70 + "\n!!  SMOKE MODE: only", SMOKE_N, "pages per set.  set SMOKE=False for the real run  !!\n" + "!" * 70)

## 2 · Inputs
**Why:** we always loop over the **manifest rows**, never over an assumed grid, because copy-move and splicing are
missing for some pages. Several rows share one file (the `anytext` and `diffste` rows for the 4 non-AI versions both
point into `shared49`), so DAF runs once per **unique `path`**: 1,522 images (1,878 rows). The md5 of
`manifest.csv` goes into `run_info.json` so Abby can check we used the same zip.

In [ ]:
# Allowed values of the 2nd folder of each manifest path
IMAGE_SETS = ["textctrl", "shared49", "anytext", "diffste"]
# Expected unique images per image set in a full run
EXPECTED_UNIQUE = {"textctrl": 970, "shared49": 356, "anytext": 98, "diffste": 98}

# Look for an already unzipped manifest (re-runs skip the unzip)
found = sorted(glob.glob(os.path.join(SETS_DIR, "**", "manifest.csv"), recursive=True))
# Unzip only when needed
if not found:
    # Find the zip in the Drive folder
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "**", "PAGES_ZIP_NAME*.zip"), recursive=True))
    # Stop if the zip is not there
    if not zips:
        # Clear message
        raise RuntimeError(f"STOP: PAGES_ZIP_NAME.zip not found in {DRIVE_DIR}")
    # Say which zip we use
    print("unzipping", zips[0], "->", SETS_DIR)
    # Open the zip
    with zipfile.ZipFile(zips[0]) as zf:
        # Extract everything to local disk
        zf.extractall(SETS_DIR)
    # Search again
    found = sorted(glob.glob(os.path.join(SETS_DIR, "**", "manifest.csv"), recursive=True))
# Stop if there is still no manifest
if not found:
    # Clear message
    raise RuntimeError("STOP: manifest.csv not found inside the zip")
# Use the shortest path (the top-level manifest) if there are several
MANIFEST = sorted(found, key=len)[0]
# Paths in the manifest are relative to its folder
MANIFEST_ROOT = os.path.dirname(MANIFEST)
# Show it
print("manifest:", MANIFEST)

# md5 of the manifest file bytes
with open(MANIFEST, "rb") as f:
    # Fingerprint for run_info.json
    manifest_md5 = hashlib.md5(f.read()).hexdigest()
# Read every column as text
man = pd.read_csv(MANIFEST, dtype=str)
# Number of manifest rows
manifest_rows_total = len(man)
# Show both
print("manifest_md5 =", manifest_md5, "| manifest_rows_total =", manifest_rows_total)
# Show the column names
print("columns:", list(man.columns))
# The path column is required
if "path" not in man.columns:
    # Clear message
    raise RuntimeError("STOP: manifest.csv has no 'path' column")

# Use forward slashes and drop a leading ./
man["path"] = man["path"].str.replace("\\", "/", regex=False).str.replace(r"^\./", "", regex=True)
# Split each path into folders
parts = man["path"].str.split("/")
# Image set = 2nd folder of the path (textctrl / shared49 / anytext / diffste)
man["image_set"] = parts.str[1]
# Manifest set (textctrl / anytext / diffste): from the 'set' column, else the image set
man["set"] = man["set"] if "set" in man.columns else man["image_set"]
# Version: from the 'version' column, else the 3rd folder
man["version"] = man["version"] if "version" in man.columns else parts.str[2]
# Format: from the 'format' column, else the file extension
man["format"] = man["format"] if "format" in man.columns else man["path"].str.rsplit(".", n=1).str[1].str.lower()
# Page name: from the 'page' column, else the file name without extension
man["page"] = man["page"] if "page" in man.columns else parts.str[-1].str.rsplit(".", n=1).str[0]

# Every image set must be one we know
bad_sets = sorted(set(man["image_set"]) - set(IMAGE_SETS))
# Stop on unknown image sets
if bad_sets:
    # It would mean a different zip layout
    raise RuntimeError(f"STOP: unexpected image sets (2nd folder of path): {bad_sets}")

# Rows per set x version x format
print("\nmanifest rows per set x version x format:")
# Count table
print(man.groupby(["set", "version", "format"]).size().unstack("format", fill_value=0).to_string())

# In SMOKE mode keep only the first SMOKE_N pages of each set
if SMOKE:
    # Chosen (set, page) pairs
    keep = []
    # One set at a time
    for s, g in man.groupby("set"):
        # First pages in name order
        for p in sorted(g["page"].unique())[:SMOKE_N]:
            # Remember the pair
            keep.append((s, p))
    # As a set for fast lookup
    keep = set(keep)
    # Keep the rows of those pages
    man_run = man[[(s, p) in keep for s, p in zip(man["set"], man["page"])]].copy()
else:
    # Full run: every row
    man_run = man.copy()

# One row per unique image file
uniq = man_run.drop_duplicates("path")[["path", "image_set", "version", "format", "page"]].reset_index(drop=True)
# Full path of each image on local disk
uniq["abs_path"] = [os.path.join(MANIFEST_ROOT, p) for p in uniq["path"]]
# Number of unique images in this run
N_UNIQUE = len(uniq)
# Show it
print(f"\nrows in this run: {len(man_run)} | unique images: {N_UNIQUE}")
# Unique images per image set
print(uniq["image_set"].value_counts().to_string())

# Full run: must be exactly 1,522 unique images
if not SMOKE:
    # Total check
    if N_UNIQUE != 1522:
        # Wrong zip
        raise RuntimeError(f"STOP: {N_UNIQUE} unique paths, expected 1522. Is this the right PAGES_ZIP_NAME zip?")
    # Per image set check
    got = uniq["image_set"].value_counts().to_dict()
    # Compare with the expected counts
    if got != EXPECTED_UNIQUE:
        # Wrong zip
        raise RuntimeError(f"STOP: unique images per set {got}, expected {EXPECTED_UNIQUE}")

# Two images must never share one result path
dup = uniq.duplicated(["image_set", "version", "format", "page"]).sum()
# Stop if they do
if dup:
    # Results would overwrite each other
    raise RuntimeError(f"STOP: {dup} images share image_set/version/format/page")
# Every image file must exist
missing = [p for p in uniq["abs_path"] if not os.path.exists(p)]
# Stop if any are missing
if missing:
    # Show a few
    raise RuntimeError(f"STOP: {len(missing)} image files missing, e.g. {missing[:5]}")
# A few example paths (never more than 20)
print("\nexample paths:", list(uniq["path"][:5]))
# Done
print("INPUTS OK")

## 3 · Environment (separate Python 3.9)
**Why:** DAF needs Python 3.9, torch 1.13 + CUDA 11.7, mmcv 2.0.0 and mmengine 0.8.5; Colab's own Python is too new.
We build a separate environment with `uv` and add packages **one at a time** (never `pip install -r requirements.txt`).
If an exact version will not install, the closest one that does is used and printed under "LOG THIS" — copy those
lines into the log at the end. A re-run skips the build if the environment already works.

In [ ]:
# Lines to copy into the error/fix log
FIXES = []
# File that marks a finished environment build
ENV_DONE = os.path.join(ENV_DIR, ".daf_env_ok")
# File that pins installed core versions so later installs cannot change them
CONSTRAINTS = os.path.join(LOCAL_DIR, "daf_constraints.txt")


# Run a shell command and return (ok, last lines of output)
def run(cmd, cwd=None):
    # Capture output so the notebook stays short
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    # Keep the end of the output for error messages
    tail = (r.stdout + r.stderr).strip().splitlines()[-3:]
    # ok + tail
    return r.returncode == 0, " | ".join(tail)


# Install one package into the 3.9 environment, trying candidates in order
def uv_install(label, candidates):
    # Each candidate is a list of uv arguments, the first is the wanted version
    for cand in candidates:
        # uv command with the constraints file
        cmd = [sys.executable, "-m", "uv", "pip", "install", "--python", ENV_PY, "-c", CONSTRAINTS] + cand
        # Try it
        ok, tail = run(cmd)
        # Success
        if ok:
            # Note when we had to fall back
            if cand is not candidates[0]:
                # Record for the log
                FIXES.append(f"{label}: wanted {candidates[0][0]}, installed {cand[0]} (wanted version would not install)")
            # Show it
            print(f"  ok   {label:<14} {cand[0]}")
            # Stop trying
            return cand[0]
        # Show the failure briefly
        print(f"  fail {label:<14} {cand[0]} -> {tail[:200]}")
    # Nothing worked
    return None


# Pin a package at the version now installed
def pin(name):
    # Ask the 3.9 Python for the version
    r = subprocess.run([ENV_PY, "-c", f"import importlib.metadata as m; print(m.version('{name}'))"],
                       capture_output=True, text=True)
    # Append to the constraints file
    with open(CONSTRAINTS, "a") as f:
        # One pin per line
        f.write(f"{name}=={r.stdout.strip()}\n")


# Wheel indexes (reachable from Colab)
TORCH_IDX = ["--extra-index-url", "https://download.pytorch.org/whl/cu117", "--index-strategy", "unsafe-best-match"]
# mmcv prebuilt wheels for CUDA 11.7 + torch 1.13 (never build mmcv from source: it takes hours)
MMCV_IDX = ["--find-links", "https://download.openmmlab.com/mmcv/dist/cu117/torch1.13/index.html", "--only-binary", "mmcv"]
# natten prebuilt wheels
NATTEN_IDX = ["--find-links", "https://shi-labs.com/natten/wheels/", "--only-binary", "natten"]

# Build the environment only once per Colab machine
if not os.path.exists(ENV_DONE):
    # Install uv into Colab's own Python
    ok, tail = run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    # Show it
    print("uv installed:", ok)
    # Create the Python 3.9 environment (uv downloads Python 3.9 itself)
    ok, tail = run([sys.executable, "-m", "uv", "venv", "--clear", "--python", "3.9", ENV_DIR])
    # Stop if that failed
    if not ok:
        # Clear message
        raise RuntimeError("STOP: could not create the Python 3.9 environment: " + tail)
    # Start with an empty constraints file
    open(CONSTRAINTS, "w").close()
    # torch + torchvision for CUDA 11.7 (fallback: PyPI build of 1.13.0, which also bundles CUDA 11.7)
    t = uv_install("torch", [["torch==1.13.0+cu117", "torchvision==0.14.0+cu117"] + TORCH_IDX,
                             ["torch==1.13.0", "torchvision==0.14.0"]])
    # Torch is essential
    if t is None:
        # Clear message
        raise RuntimeError("STOP: torch 1.13 did not install")
    # Pin torch and torchvision
    pin("torch"); pin("torchvision")
    # numpy 1.21 as in requirements.txt
    uv_install("numpy", [["numpy==1.22.4"], ["numpy==1.22.0"]])
    # Pin numpy so nothing upgrades it
    pin("numpy")
    # setuptools (mmengine imports pkg_resources; uv environments start without it)
    uv_install("setuptools", [["setuptools==61.2.0"], ["setuptools<70"]])
    # yapf 0.40.1: newer yapf breaks mmengine 0.8.5 (FormatCode 'verify' error)
    uv_install("yapf", [["yapf==0.40.1"]])
    # OpenCV (installed before mmcv so mmcv does not pull a numpy-2 OpenCV)
    uv_install("opencv", [["opencv-python==4.8.0.76", "opencv-python-headless==4.8.0.76"]])
    # mmengine
    uv_install("mmengine", [["mmengine==0.8.5"]])
    # mmcv with compiled CUDA ops (RoIAlign, nms)
    m = uv_install("mmcv", [["mmcv==2.0.0"] + MMCV_IDX, ["mmcv==2.0.1"] + MMCV_IDX])
    # mmcv is essential
    if m is None:
        # Clear message
        raise RuntimeError("STOP: no prebuilt mmcv 2.0.x wheel installed")
    # timm
    uv_install("timm", [["timm==0.6.13"]])
    # The other packages the DAF mmdet imports when it loads, one at a time
    for spec in ["Pillow==9.0.1", "scipy==1.7.3", "scikit-image==0.19.2", "Shapely==2.0.6",
                 "pycocotools==2.0.7", "terminaltables==3.1.10", "six==1.16.0", "rich==13.8.1",
                 "matplotlib==3.5.1", "tqdm==4.64.0"]:
        # Exact version first, then any version that fits the pins
        uv_install(spec.split("==")[0], [[spec], [spec.split("==")[0]]])
    # numba: 0.58.1 needs numpy>=1.22, so 0.56.4 is the closest one that works with numpy 1.21
    uv_install("numba", [["numba==0.58.1"], ["numba==0.56.4"]])
    # albumentations
    uv_install("albumentations", [["albumentations==1.3.1"]])
    # natten: only the NAT backbone needs it (not imported by the Faster R-CNN Swin configs), so a failure is OK
    n = uv_install("natten", [["natten==0.17.1"] + NATTEN_IDX, ["natten==0.14.6+torch1130cu117"] + NATTEN_IDX])
    # Note when natten is missing
    if n is None:
        # Record for the log
        FIXES.append("natten: no wheel for torch 1.13 installed; not needed (DAF Faster R-CNN uses Swin, nat.py is never imported)")

# Clone the OSTF repo once
if not os.path.isdir(REPO_DIR):
    # Shallow clone is enough
    ok, tail = run(["git", "clone", "--depth", "1", "https://github.com/qcf-568/OSTF.git", REPO_DIR])
    # Stop if it failed
    if not ok:
        # Clear message
        raise RuntimeError("STOP: git clone failed: " + tail)
# README inference step: mv mmdet mmdet_train; unzip mmdet_test.zip
if not os.path.isdir(os.path.join(REPO_DIR, "mmdet_train")):
    # Keep the training copy aside
    shutil.move(os.path.join(REPO_DIR, "mmdet"), os.path.join(REPO_DIR, "mmdet_train"))
    # Unpack the test copy of mmdet
    with zipfile.ZipFile(os.path.join(REPO_DIR, "mmdet_test.zip")) as zf:
        # Into the repo folder
        zf.extractall(REPO_DIR)
    # Remove stale compiled files that came inside the zip
    for d in glob.glob(os.path.join(REPO_DIR, "mmdet", "**", "__pycache__"), recursive=True):
        # Not needed
        shutil.rmtree(d, ignore_errors=True)

# Check script run with the 3.9 Python: import everything DAF needs and touch the GPU
CHECK = r"""
# Tools
import sys, json
# Make the repo's mmdet and suploss.py importable
sys.path.insert(0, sys.argv[1])
# Core packages
import torch, torchvision, numpy, mmcv, mmengine, timm, cv2
# mmcv's compiled CUDA ops
from mmcv.ops import nms
# The DAF copy of mmdet (registers all model parts)
import mmdet, mmdet.models
# The model registry
from mmdet.registry import MODELS
# One box on the GPU
b = torch.tensor([[0., 0., 10., 10.]], device="cuda")
# Run a compiled CUDA op once to prove the GPU build works
nms(b, torch.tensor([0.9], device="cuda"), 0.5)
# Versions as one JSON line
print(json.dumps({"python": sys.version.split()[0], "torch": torch.__version__, "torch_cuda": torch.version.cuda,
  "torchvision": torchvision.__version__, "mmcv": mmcv.__version__, "mmengine": mmengine.__version__,
  "mmdet": mmdet.__version__, "timm": timm.__version__, "numpy": numpy.__version__, "opencv": cv2.__version__,
  "gpu": torch.cuda.get_device_name(0), "daf_head": MODELS.get("DFPNCMap3") is not None,
  "daf_detector": MODELS.get("FTDFPNTextureMaskRCNN") is not None}))
"""
# Run the check
r = subprocess.run([ENV_PY, "-c", CHECK, REPO_DIR], capture_output=True, text=True, cwd=LOCAL_DIR)
# Stop with the error if it failed
if r.returncode != 0:
    # Show the last part of the error
    raise RuntimeError("STOP: environment check failed:\n" + r.stderr[-3000:])
# Versions are the last printed line
ENV_VERSIONS = json.loads(r.stdout.strip().splitlines()[-1])
# Show them
for k, v in ENV_VERSIONS.items():
    # One per line
    print(f"  {k:<13} {v}")
# The DAF classes must be registered
if not (ENV_VERSIONS["daf_head"] and ENV_VERSIONS["daf_detector"]):
    # Clear message
    raise RuntimeError("STOP: DAF classes not registered - is mmdet_test.zip unzipped?")
# torch 1.13 has no kernels for H100 (sm_90)
if "H100" in ENV_VERSIONS["gpu"]:
    # Warn
    print("WARNING: torch 1.13 does not support H100. Pick a T4, L4 or A100 runtime.")
# Mark the build as done
open(ENV_DONE, "w").close()
# Show fallbacks to log
for line in FIXES:
    # One per line
    print("LOG THIS:", line)
# Done
print("ENV OK")

## 4 · Setup check (weights ↔ config, tampered score)
**Why:** a weight file loaded into the wrong model fails silently, so we require **0 missing and 0 unexpected keys**.
All three weights use the Faster R-CNN DAF configs `configs/fasterrcnn_x<editor>.py` (Texture Jitter pre-training,
RoI head `DFPNCMap3`); they match the files in `pths/dfpn/` of the official release. `pths/cascade/` weights would
fail this check. In `mmdet_test`'s `dfpncmap3.py`, `predict_mask` (near line 548/574) returns
`F.softmax(self.fc(self.convert(m)-g), 1)`: **column 1 = tampered**, column 0 = real (the repo's evaluator files
`argmax == 1` as fake). Only boxes with text-detection score > 0.5 are returned (`zl`), and we set `max_per_img=1000`.

In [ ]:
# The DAF script, run by the Python 3.9 environment
DAF_SCRIPT = r"""
# Command-line options, file handling, timing
import argparse, json, os, sys, time
# Arrays and images
import numpy as np
import cv2
# PyTorch
import torch

# Read the options
ap = argparse.ArgumentParser()
# Folder of the OSTF repo (with mmdet_test unzipped)
ap.add_argument("--repo")
# DAF config file
ap.add_argument("--config")
# Weight file
ap.add_argument("--weights")
# JSON list of jobs (one job = one page at one size, made of 1+ input images)
ap.add_argument("--jobs")
# check = print keys + scores only, run = write result JSONs
ap.add_argument("--mode", default="run")
# Jobs still waiting for later weights (only used for the time estimate)
ap.add_argument("--later_jobs", type=int, default=0)
# Parse them
a = ap.parse_args()

# Make the repo's mmdet and suploss.py importable
sys.path.insert(0, a.repo)
# Config reader
from mmengine.config import Config
# Registry scope helper
from mmengine.registry import init_default_scope
# Model registry and data container
from mmdet.registry import MODELS
from mmdet.structures import DetDataSample
# Register all DAF model parts
import mmdet.models
# Resize helper used by the config's Resize step
import mmcv

# Load the training config
cfg = Config.fromfile(a.config)
# README inference step: max_per_img=1000
cfg.model.test_cfg.rcnn.max_per_img = 1000
# Use mmdet names
init_default_scope("mmdet")
# Build the DAF model
model = MODELS.build(cfg.model)
# Read the weight file
ck = torch.load(a.weights, map_location="cpu")
# The weights may sit under 'state_dict'
sd = ck.get("state_dict", ck)
# Drop a 'module.' prefix left by multi-GPU training
sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}
# Load and collect key differences
res = model.load_state_dict(sd, strict=False)
# Report the key check as one JSON line
print("KEYS", json.dumps({"missing": len(res.missing_keys), "unexpected": len(res.unexpected_keys),
                          "missing_first": res.missing_keys[:5], "unexpected_first": res.unexpected_keys[:5]}), flush=True)
# GPU, evaluation mode
model = model.cuda().eval()

# Run DAF on one image file, return boxes (in that image's pixels) and [real, tampered] probabilities
def run_one(path):
    # Read as 3-channel BGR like the config's LoadImageFromFile (ignore EXIF rotation: raw pixels)
    img = cv2.imread(path, cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    # Original size
    h, w = img.shape[:2]
    # Config test step: Resize scale=(1024,1536), keep_ratio=True
    img_r = mmcv.imrescale(img, (1024, 1536), interpolation="bilinear", backend="cv2")
    # Resized size
    nh, nw = img_r.shape[:2]
    # Channels first tensor
    t = torch.from_numpy(np.ascontiguousarray(img_r.transpose(2, 0, 1)))
    # Image info the model needs (scale_factor = (w_scale, h_scale) as mmdet uses it)
    ds = DetDataSample(metainfo=dict(img_id=0, img_path=path, ori_shape=(h, w), img_shape=(nh, nw),
                                     scale_factor=(nw / w, nh / h)))
    # Inference without gradients
    with torch.no_grad():
        # test_step returns one (boxes, probs) tuple per image, boxes scaled back to the original size
        boxes, probs = model.test_step(dict(inputs=[t], data_samples=[ds]))[0]
    # Box objects -> plain tensor
    if hasattr(boxes, "tensor"):
        # Unwrap
        boxes = boxes.tensor
    # No boxes: the model returns an empty container instead of a tensor
    if (not torch.is_tensor(probs)) or probs.numel() == 0:
        # Empty arrays
        return np.zeros((0, 4)), np.zeros((0, 2)), w, h
    # To numpy
    return boxes.float().cpu().numpy(), probs.float().cpu().numpy(), w, h

# Read the job list
jobs = json.load(open(a.jobs))
# Check mode: 3 images, print results, write nothing
if a.mode == "check":
    # One job at a time
    for j in jobs:
        # Run the single input
        b, p, w, h = run_one(j["inputs"][0]["path"])
        # Highest tampered probability (column 1)
        top = float(p[:, 1].max()) if len(p) else 0.0
        # One JSON line per image
        print("CHECK", json.dumps({"name": j["name"], "n_boxes": len(b), "max_tampered": round(top, 4),
                                   "n_over_thresh": int((p[:, 1] >= 0.5).sum()) if len(p) else 0,
                                   "first_probs_[real,tampered]": np.round(p[:3], 3).tolist()}), flush=True)
    # Done
    sys.exit(0)

# Run mode: count pages done and time them
done, t_start, est_printed = 0, time.time(), False
# One job (page at one size) at a time
for j in jobs:
    # Resumable: skip pages already done
    if os.path.exists(j["out"]):
        # Next page
        continue
    # Results of each input piece
    pieces = []
    # Every input image of this page (1, or several for "full")
    for inp in j["inputs"]:
        # Wait for the GPU before timing
        torch.cuda.synchronize()
        # Start the clock
        t0 = time.time()
        # Run DAF
        b, p, w, h = run_one(inp["path"])
        # Wait for the GPU again
        torch.cuda.synchronize()
        # Store boxes, probs, position and time
        pieces.append({"x0": inp["x0"], "y0": inp["y0"], "in_w": w, "in_h": h, "boxes": b.tolist(),
                       "probs": p.tolist(), "seconds": time.time() - t0})
    # Make the output folder
    os.makedirs(os.path.dirname(j["out"]), exist_ok=True)
    # Write to a temp file first so a disconnect never leaves half a file
    with open(j["out"] + ".tmp", "w") as f:
        # Raw result of this page
        json.dump({"pieces": pieces, "seconds": sum(x["seconds"] for x in pieces)}, f)
    # Rename into place
    os.replace(j["out"] + ".tmp", j["out"])
    # Count it
    done += 1
    # After the first 20 pages, print a time estimate
    if done == 20 and not est_printed:
        # Seconds per page so far
        per = (time.time() - t_start) / 20
        # Pages left in this run and later weights
        left = len(jobs) - 20 + a.later_jobs
        # Show the estimate
        print(f"TIMING first 20 pages: {per:.2f} s/page -> about {per * left / 60:.0f} min for the {left} pages left", flush=True)
        # Only once
        est_printed = True
    # Progress every 200 pages
    if done % 200 == 0:
        # Short progress line
        print(f"  {done}/{len(jobs)} pages done", flush=True)
# Summary
print(f"RUN DONE: {done} new pages in {time.time() - t_start:.0f} s", flush=True)
"""
# Write the script to local disk
with open(SCRIPT, "w") as f:
    # Script text
    f.write(DAF_SCRIPT)

# Find each weight file in the Drive folder and its config
WEIGHT_FILES, CONFIGS = {}, {}
# One weight at a time
for w in WEIGHTS:
    # Search the Drive folder for <weight>.pth
    hits = sorted(glob.glob(os.path.join(DRIVE_DIR, "**", f"{w}.pth"), recursive=True))
    # Stop if missing
    if not hits:
        # Clear message
        raise RuntimeError(f"STOP: {w}.pth not found in {DRIVE_DIR}")
    # Copy to local disk once (faster to load)
    local_w = os.path.join(LOCAL_DIR, "daf_weights", f"{w}.pth")
    # Only if not copied yet
    if not os.path.exists(local_w):
        # Make the folder
        os.makedirs(os.path.dirname(local_w), exist_ok=True)
        # Copy
        shutil.copy(hits[0], local_w)
    # Remember the weight file
    WEIGHT_FILES[w] = local_w
    # Matching Faster R-CNN DAF config (Texture Jitter pre-trained)
    CONFIGS[w] = os.path.join(REPO_DIR, "configs", f"fasterrcnn_x{w}.py")
    # Show the pair (names only)
    print(f"{w}: weight {os.path.basename(hits[0])} ({os.path.getsize(hits[0]) / 1e6:.0f} MB) <-> config {os.path.basename(CONFIGS[w])}")

# Three test images: one from each image set if possible
check_rows = uniq.groupby("image_set").head(1).head(3)
# Jobs for the check (original page, as DAF resizes it itself)
check_jobs = [{"name": r.path, "inputs": [{"path": r.abs_path, "x0": 0, "y0": 0}]} for r in check_rows.itertuples()]
# Save the check job list
CHECK_JOBS = os.path.join(LOCAL_DIR, "jobs_check.json")
# Write it
with open(CHECK_JOBS, "w") as f:
    # JSON list
    json.dump(check_jobs, f)

# Check each weight
for w in WEIGHTS:
    # Run the script in check mode with the 3.9 Python
    r = subprocess.run([ENV_PY, SCRIPT, "--repo", REPO_DIR, "--config", CONFIGS[w], "--weights", WEIGHT_FILES[w],
                        "--jobs", CHECK_JOBS, "--mode", "check"], capture_output=True, text=True, cwd=LOCAL_DIR)
    # Stop on errors
    if r.returncode != 0:
        # Last part of the error
        raise RuntimeError(f"STOP: setup check failed for {w}:\n" + r.stderr[-3000:])
    # Lines the script printed
    lines = r.stdout.splitlines()
    # The key report
    keys = json.loads([x for x in lines if x.startswith("KEYS ")][0][5:])
    # Show it
    print(f"\n[{w}] missing keys: {keys['missing']} | unexpected keys: {keys['unexpected']}")
    # Keys must match exactly
    if keys["missing"] or keys["unexpected"]:
        # Probably a Cascade weight or wrong file
        raise RuntimeError(f"STOP: {w}.pth does not match {os.path.basename(CONFIGS[w])}: {keys}. "
                           "Use the Faster R-CNN weights from pths/dfpn/, not pths/cascade/.")
    # Per-image results
    for x in [x for x in lines if x.startswith("CHECK ")]:
        # Show each
        print("  ", x[6:])
# Done
print("\nSETUP CHECK OK (tampered score = softmax column 1)")

## 5 · Prepare inputs
**Why:** the three sizes test DAF at different resolutions. `shrink512`: long side 512 px (area interpolation).
`default`: the page unchanged (DAF fits it inside 1024 × 1536 itself). `full`: pieces of 1536 px along the page's long
side × 1024 px along its short side from (0,0), so each piece reaches DAF at native resolution; edges padded white.
Everything goes to local disk; the piece table records where each piece sits so boxes can be put back.

In [ ]:
# Image tools for the Colab side
import cv2

# Page sizes, shrunk copies and piece table
page_rows, piece_rows = [], []
# Count newly written files
n_new = 0
# One unique image at a time
for r in uniq.itertuples():
    # Folder part shared by all outputs of this image
    rel = os.path.join(r.image_set, r.version, r.format)
    # Shrunk copy path
    small = os.path.join(INPUT_DIR, "shrink512", rel, r.page + ".png")
    # Read the page as 3-channel BGR (raw pixels, like the DAF script)
    img = cv2.imread(r.abs_path, cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    # Page height and width
    H, W = img.shape[:2]
    # Scale so the long side becomes 512
    s = SHRINK_LONG / max(H, W)
    # Shrunk size (at least 1 px)
    sw, sh = max(1, round(W * s)), max(1, round(H * s))
    # Write the shrunk copy if not there yet
    if not os.path.exists(small):
        # Make the folder
        os.makedirs(os.path.dirname(small), exist_ok=True)
        # Area interpolation, lossless PNG
        cv2.imwrite(small, cv2.resize(img, (sw, sh), interpolation=cv2.INTER_AREA))
        # Count it
        n_new += 1
    # Landscape pages get landscape pieces, portrait pages portrait pieces
    pw, ph = (PIECE_LONG, PIECE_SHORT) if W >= H else (PIECE_SHORT, PIECE_LONG)
    # Number of pieces
    n_pieces = 0
    # Rows of pieces from the top
    for y0 in range(0, H, ph):
        # Columns of pieces from the left
        for x0 in range(0, W, pw):
            # Piece file path
            ppath = os.path.join(INPUT_DIR, "full", rel, f"{r.page}__y{y0}_x{x0}.png")
            # Write the piece if not there yet
            if not os.path.exists(ppath):
                # White canvas of the full piece size (this is the padding)
                piece = np.full((ph, pw, 3), 255, dtype=np.uint8)
                # Part of the page inside this piece
                crop = img[y0:y0 + ph, x0:x0 + pw]
                # Paste it at the top-left
                piece[:crop.shape[0], :crop.shape[1]] = crop
                # Make the folder
                os.makedirs(os.path.dirname(ppath), exist_ok=True)
                # Lossless PNG
                cv2.imwrite(ppath, piece)
                # Count it
                n_new += 1
            # Remember where it sits
            piece_rows.append({"path": r.path, "piece": ppath, "x0": x0, "y0": y0, "pw": pw, "ph": ph})
            # Count pieces
            n_pieces += 1
    # Remember the page info
    page_rows.append({"path": r.path, "W": W, "H": H, "small": small, "sw": sw, "sh": sh, "n_pieces": n_pieces})

# Page table
pages = uniq.merge(pd.DataFrame(page_rows), on="path")
# Piece table
pieces = pd.DataFrame(piece_rows)
# Keep the piece table on Drive
pieces.to_csv(os.path.join(RAW_DIR, "pieces_smoke.csv" if SMOKE else "pieces.csv"), index=False)
# Summary
print(f"pages: {len(pages)} | full-size pieces: {len(pieces)} | new files written: {n_new}")
# Page size range
print(f"page width {pages.W.min()}-{pages.W.max()} px, height {pages.H.min()}-{pages.H.max()} px")
# Pieces per page
print("pieces per page:", pages.n_pieces.value_counts().sort_index().to_dict())
# Done
print("PREPARE OK")

## 6 · Run DAF
**Why:** one script call per weight loads the model once and processes every page at all 3 sizes. Each page's raw
boxes are written to `raw_daf/` on Drive as soon as they are ready, and pages whose file exists are skipped, so a
re-run after a disconnect continues where it stopped. The script prints a time estimate after the first 20 pages.

In [ ]:
# Raw result file of one weight x size x image
def raw_path(w, size, r):
    # raw_daf/daf_<weight>/<size>/<image set>/<version>/<format>/<page>.json
    return os.path.join(RAW_DIR, f"daf_{w}", size, r.image_set, r.version, r.format, r.page + ".json")


# Pieces of each page, grouped once
pieces_by_path = {p: g for p, g in pieces.groupby("path")}
# The job list is the same for every weight except the output paths
def make_jobs(w):
    # Jobs for this weight
    jobs = []
    # Every size
    for size in SIZES:
        # Every unique page
        for r in pages.itertuples():
            # Inputs: the shrunk copy, the page itself, or its pieces
            if size == "shrink512":
                # One input
                inputs = [{"path": r.small, "x0": 0, "y0": 0}]
            elif size == "default":
                # One input: the original file
                inputs = [{"path": r.abs_path, "x0": 0, "y0": 0}]
            else:
                # All pieces with their positions
                inputs = [{"path": p.piece, "x0": int(p.x0), "y0": int(p.y0)} for p in pieces_by_path[r.path].itertuples()]
            # One job
            jobs.append({"name": r.path, "out": raw_path(w, size, r), "inputs": inputs})
    # All jobs
    return jobs


# Jobs still to do for each weight
todo = {w: [j for j in make_jobs(w) if not os.path.exists(j["out"])] for w in WEIGHTS}
# Show the plan
print({w: f"{len(todo[w])} of {len(pages) * len(SIZES)} pages to run" for w in WEIGHTS})
# Run each weight
for i, w in enumerate(WEIGHTS):
    # Nothing left for this weight
    if not todo[w]:
        # Skip it
        print(f"[{w}] all done already")
        # Next weight
        continue
    # Job file on local disk
    jpath = os.path.join(LOCAL_DIR, f"jobs_{w}.json")
    # Write it
    with open(jpath, "w") as f:
        # JSON list
        json.dump(todo[w], f)
    # Pages waiting for later weights (for the estimate)
    later = sum(len(todo[x]) for x in WEIGHTS[i + 1:])
    # Show the start
    print(f"[{w}] running {len(todo[w])} pages ...")
    # Start the script and stream its output
    p = subprocess.Popen([ENV_PY, "-u", SCRIPT, "--repo", REPO_DIR, "--config", CONFIGS[w], "--weights", WEIGHT_FILES[w],
                          "--jobs", jpath, "--mode", "run", "--later_jobs", str(later)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=LOCAL_DIR)
    # Keep the last lines for error messages
    last = []
    # Read output line by line
    for line in p.stdout:
        # Remember it
        last = (last + [line.rstrip()])[-30:]
        # Show only our own lines
        if line.startswith(("TIMING", "  ", "RUN DONE", "KEYS")):
            # Print it
            print("  ", line.rstrip())
    # Wait for the end
    p.wait()
    # Stop on errors
    if p.returncode != 0:
        # Show the end of the output
        raise RuntimeError(f"STOP: DAF run failed for {w} (re-run to resume):\n" + "\n".join(last))

# Count finished raw files
n_raw = {w: sum(os.path.exists(j["out"]) for j in make_jobs(w)) for w in WEIGHTS}
# Show them
print("raw page results:", n_raw, "| expected per weight:", len(pages) * len(SIZES))
# Done
print("RUN OK")

## 7 · STANDARD results
**Why:** Abby scores every detector the same way, so each result is an 8-bit greyscale PNG the **same size as the
page**, each box filled with `round(255 × tampered)` (round half up, so "≥ 128" means probability ≥ 0.5), the
maximum where boxes overlap, 0 elsewhere. `full` boxes are shifted by their piece position and cut at the page edge;
`shrink512` boxes are scaled back up. An image with no box still gets an all-zero map. `boxes.csv` and `run_log.csv`
are rebuilt from the raw files every run, so re-runs never duplicate rows.

In [ ]:
# Write 8-bit PNG maps
from PIL import Image

# Totals for the summary
summary = []
# One weight at a time
for w in WEIGHTS:
    # Detector folder in results/
    det_dir = os.path.join(RESULTS_DIR, f"daf_{w}")
    # Rows of boxes.csv and run_log.csv
    box_rows, log_rows = [], []
    # Every size
    for size in SIZES:
        # Every unique page
        for r in pages.itertuples():
            # Result map path
            out = os.path.join(det_dir, size, r.image_set, r.version, r.format, r.page + ".png")
            # Raw DAF result of this page
            rp = raw_path(w, size, r)
            # Raw result missing: DAF did not finish this page
            if not os.path.exists(rp):
                # Log it (section 8 will stop)
                log_rows.append({"image": r.path, "size": size, "set": r.image_set, "version": r.version,
                                 "format": r.format, "page": r.page, "seconds": "", "max_value": "", "status": "missing_raw"})
                # Next page
                continue
            # Read the raw result
            raw = json.load(open(rp))
            # Empty map of the page size
            m = np.zeros((r.H, r.W), dtype=np.uint8)
            # Every input piece of this page
            for pc in raw["pieces"]:
                # Every box with its probabilities
                for (x1, y1, x2, y2), (p_real, p_tamp) in zip(pc["boxes"], pc["probs"]):
                    # shrink512: scale back up to page size
                    if size == "shrink512":
                        # Separate x and y factors
                        fx, fy = r.W / r.sw, r.H / r.sh
                        # Scale the box
                        x1, x2, y1, y2 = x1 * fx, x2 * fx, y1 * fy, y2 * fy
                    # full: shift by the piece position
                    if size == "full":
                        # Shift the box
                        x1, x2, y1, y2 = x1 + pc["x0"], x2 + pc["x0"], y1 + pc["y0"], y2 + pc["y0"]
                    # Cut at the page edge (removes the white padding)
                    x1, x2 = min(max(x1, 0), r.W), min(max(x2, 0), r.W)
                    # Same for y
                    y1, y2 = min(max(y1, 0), r.H), min(max(y2, 0), r.H)
                    # Keep the raw box (page coordinates) for boxes.csv
                    box_rows.append({"image_set": r.image_set, "version": r.version, "format": r.format, "size": size,
                                     "page": r.page, "x1": round(x1, 2), "y1": round(y1, 2), "x2": round(x2, 2),
                                     "y2": round(y2, 2), "tampered_score": round(p_tamp, 6)})
                    # Map value: round half up
                    v = int(math.floor(255 * p_tamp + 0.5))
                    # Whole pixels covered by the box
                    c1, c2, r1, r2 = int(math.floor(x1)), int(math.ceil(x2)), int(math.floor(y1)), int(math.ceil(y2))
                    # Skip boxes with no area
                    if c2 <= c1 or r2 <= r1:
                        # Next box
                        continue
                    # Keep the maximum where boxes overlap
                    m[r1:r2, c1:c2] = np.maximum(m[r1:r2, c1:c2], v)
            # The map must have the page size
            assert m.shape == (r.H, r.W) and m.dtype == np.uint8
            # Write it (also when all zero) unless it already exists
            if not os.path.exists(out):
                # Make the folder
                os.makedirs(os.path.dirname(out), exist_ok=True)
                # Write to a temp name first
                Image.fromarray(m, mode="L").save(out + ".tmp.png")
                # Rename into place
                os.replace(out + ".tmp.png", out)
            # One run_log row per image and size
            log_rows.append({"image": r.path, "size": size, "set": r.image_set, "version": r.version, "format": r.format,
                             "page": r.page, "seconds": round(raw["seconds"], 4), "max_value": int(m.max()),
                             "status": "ok" if m.max() > 0 else "no_boxes"})
    # Columns of boxes.csv
    box_cols = ["image_set", "version", "format", "size", "page", "x1", "y1", "x2", "y2", "tampered_score"]
    # Write boxes.csv fresh
    pd.DataFrame(box_rows, columns=box_cols).to_csv(os.path.join(det_dir, "boxes.csv"), index=False)
    # Columns of run_log.csv
    log_cols = ["image", "size", "set", "version", "format", "page", "seconds", "max_value", "status"]
    # The run log
    log = pd.DataFrame(log_rows, columns=log_cols)
    # Write run_log.csv fresh
    log.to_csv(os.path.join(det_dir, "run_log.csv"), index=False)
    # Numeric max values
    mv = pd.to_numeric(log["max_value"], errors="coerce")
    # Summary per size
    for size in SIZES:
        # Rows of this size
        sel = log["size"] == size
        # One summary line
        summary.append({"weight": w, "size": size, "maps": int((log[sel].status != "missing_raw").sum()),
                        "no_box": int((log[sel].status == "no_boxes").sum()),
                        f"flagged(>= {round(255 * THRESH)})": int((mv[sel] >= round(255 * THRESH)).sum()),
                        "mean_s": round(pd.to_numeric(log[sel]["seconds"], errors="coerce").mean(), 3)})
    # Show the box count
    print(f"[{w}] boxes: {len(box_rows)} | run_log rows: {len(log)}")
# Summary table
print(pd.DataFrame(summary).to_string(index=False))
# Done
print("STANDARD RESULTS WRITTEN")

## 8 · Checks
**Why:** Abby's scoring notebook stops if any image lacks a map at any of the 3 sizes, so we check first. Every
unique manifest image must have a result file for every weight and size (1,522 each; 4,566 per weight; 13,698 in
all), and `run_log.csv` must have 1,522 × 3 rows per weight. In SMOKE mode the expected numbers come from the SMOKE
subset instead.

In [ ]:
# Expected unique images (1,522 in a full run)
expected = N_UNIQUE if SMOKE else 1522
# Problems found
problems = []
# Count table rows
count_rows = []
# One weight at a time
for w in WEIGHTS:
    # Detector folder
    det_dir = os.path.join(RESULTS_DIR, f"daf_{w}")
    # Every size
    for size in SIZES:
        # Does each unique image have its map?
        have = [os.path.exists(os.path.join(det_dir, size, r.image_set, r.version, r.format, r.page + ".png"))
                for r in pages.itertuples()]
        # Number present
        n_have = sum(have)
        # Must equal the expected count
        if n_have != expected:
            # Note the problem with examples
            problems.append(f"daf_{w}/{size}: {n_have} maps, expected {expected}; missing e.g. "
                            f"{list(pages.path[[not h for h in have]][:5])}")
        # Counts per image set x version x format
        t = pages.assign(have=have).groupby(["image_set", "version", "format"])["have"].sum()
        # Keep as a column
        count_rows.append(t.rename(f"{w}/{size}"))
    # The run log
    log = pd.read_csv(os.path.join(det_dir, "run_log.csv"))
    # Must have expected x 3 rows
    if len(log) != expected * len(SIZES):
        # Note it
        problems.append(f"daf_{w}/run_log.csv: {len(log)} rows, expected {expected * len(SIZES)}")
    # No missing raw results
    if (log.status == "missing_raw").any():
        # Note it
        problems.append(f"daf_{w}: {(log.status == 'missing_raw').sum()} images without a DAF result")
    # Each image x size exactly once
    if log.duplicated(["image", "size"]).any():
        # Note it
        problems.append(f"daf_{w}/run_log.csv has duplicate image/size rows")

# Count table per image set x version x format
print(pd.concat(count_rows, axis=1).to_string())
# Total maps
print(f"\nmaps expected per weight & size: {expected} | per weight: {expected * len(SIZES)} | all: {expected * len(SIZES) * len(WEIGHTS)}")
# Stop if anything is wrong
if problems:
    # All problems at once
    raise RuntimeError("STOP:\n" + "\n".join(problems))
# Done
print("CHECKS OK: every unique image has a map at all 3 sizes for all 3 weights")

## 9 · run_info + zip
**Why:** `run_info.json` inside each detector folder tells Abby's notebook exactly what produced the maps (detector,
weight, manifest fingerprint, sizes, versions, speed); the folder name `daf_<weight>` is what her notebook trusts.
Then `results/` is zipped into `daf_results.zip` in the Drive folder — share that file's Drive link with Abby.

In [ ]:
# One run_info.json per weight
for w in WEIGHTS:
    # Detector folder
    det_dir = os.path.join(RESULTS_DIR, f"daf_{w}")
    # Seconds per image from the run log
    log = pd.read_csv(os.path.join(det_dir, "run_log.csv"))
    # Mean seconds per image and size
    sec = {s: round(float(log[log["size"] == s]["seconds"].mean()), 4) for s in SIZES}
    # Everything Abby needs to know
    info = {"detector": "daf", "weights": w, "manifest_md5": manifest_md5, "manifest_rows_total": manifest_rows_total,
            "n_images": 1522 if not SMOKE else N_UNIQUE, "smoke": SMOKE, "sizes": SIZES,
            "weight_file": os.path.basename(WEIGHT_FILES[w]), "weight_source": WEIGHT_SOURCE,
            "config": os.path.basename(CONFIGS[w]), "model": "Faster R-CNN DAF (DFPNCMap3), max_per_img=1000",
            "tampered_score": "softmax column 1 of DFPNCMap3.predict_mask",
            "daf_test_resize": [1024, 1536], "piece_long": PIECE_LONG, "piece_short": PIECE_SHORT,
            "shrink_long": SHRINK_LONG, "map_value": "round(255 * tampered), max over overlapping boxes",
            "versions": {k: ENV_VERSIONS[k] for k in ["python", "torch", "torch_cuda", "mmcv", "mmengine", "mmdet", "timm", "gpu"]},
            "seconds_per_image": sec}
    # Write it
    with open(os.path.join(det_dir, "run_info.json"), "w") as f:
        # Pretty JSON
        json.dump(info, f, indent=2)
    # Show the key facts
    print(f"daf_{w}/run_info.json: weights={w}, n_images={info['n_images']}, s/image={sec}")

# Zip on local disk first (faster), then copy to Drive
local_zip = os.path.join(LOCAL_DIR, "daf_results.zip")
# Final place on Drive
drive_zip = os.path.join(DRIVE_DIR, "daf_results_SMOKE.zip" if SMOKE else "daf_results.zip")
# Number of files zipped
n_files = 0
# Create the zip (PNGs are already compressed, so store them; deflate the CSV/JSON)
with zipfile.ZipFile(local_zip, "w") as zf:
    # Walk results/
    for root, dirs, files in os.walk(RESULTS_DIR):
        # Same order every time
        dirs.sort()
        # Each file
        for name in sorted(files):
            # Skip leftover temp files
            if ".tmp" in name:
                # Next file
                continue
            # Full path
            full = os.path.join(root, name)
            # Path inside the zip starts with results/
            arc = os.path.relpath(full, DRIVE_DIR)
            # Compression type
            how = zipfile.ZIP_STORED if name.endswith(".png") else zipfile.ZIP_DEFLATED
            # Add it
            zf.write(full, arc, compress_type=how)
            # Count it
            n_files += 1
# Copy to Drive
shutil.copy(local_zip, drive_zip)
# Show size and path
print(f"\n{n_files} files -> {drive_zip} ({os.path.getsize(drive_zip) / 1e6:.1f} MB)")
# Reminder
print("SMOKE zip - do not send. Set SMOKE=False and Run all." if SMOKE else "Send Abby the Drive link to daf_results.zip")

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

## Error / fix log
Format: `- YYYY-MM-DD HH:MM — what — why / fix`. Add every error you hit and every "LOG THIS" line from section 3.

- 2026-11-10 12:00 — repo `PackDetInputs` (mmdet_test) reads `gt_bboxes` even in test mode — our pages have no annotations / the DAF script builds the model input itself with the same resize as the config (`mmcv.imrescale` to fit 1024×1536, keep ratio, BGR, `scale_factor=(w_scale,h_scale)`) and calls `model.test_step`.
- 2026-11-10 12:00 — `init_detector` would build the config's test datasets (`mmacc/*.pk`, not present) — build the model with `MODELS.build(cfg.model)` and `load_state_dict(strict=False)` so missing/unexpected keys can be printed.
- 2026-11-10 12:00 — DAF `predict` returns raw `(boxes, probs)` tuples, and an empty container when there is no box — the script treats a non-tensor as "no boxes" (all-zero map).
- 2026-11-10 12:00 — numba 0.58.1 (requirements.txt) needs numpy ≥ 1.22, but numpy 1.21 is pinned — the env cell falls back to numba 0.56.4.
- 2026-11-10 12:00 — mmengine 0.8.5 breaks with yapf ≥ 0.40.2 (`FormatCode() got an unexpected keyword argument 'verify'`) — pin yapf 0.40.1; uv environments have no setuptools, so setuptools 61.2.0 is installed.
- 2026-11-10 12:00 — natten 0.17.1 has no torch 1.13 wheel — not needed: the Faster R-CNN configs use Swin, and `nat.py` is not imported by `mmdet/models/backbones/__init__.py`.
- 2026-11-10 12:00 — `suploss.py` sits in the repo root and is imported by mmdet — the script puts the repo root on `sys.path`.
- 2026-11-10 12:00 — EXIF rotation could make cv2 and PIL disagree on page size — both the prep and the DAF script read with `IMREAD_IGNORE_ORIENTATION` (raw pixels, same as the masks).